# 1. PHOTO MATCH com FaceNet: comparação entre documento e selfie

Esta aplicação compara a foto de um documento com uma foto atual do usuário.

O objetivo é simular uma verificação de identidade, parecida com processos de cadastro bancário, KYC e validação de conta.

A comparação não é feita pixel por pixel. O sistema detecta o rosto, gera um vetor numérico com FaceNet e compara a distância entre os vetores.

# 2. Contextualização da aplicação

O problema resolvido é verificar se duas fotos provavelmente pertencem à mesma pessoa.

Esse tipo de solução é usado em:

- bancos digitais;
- cadastros online;
- validação de documentos;
- KYC;
- prova de vida;
- controle de acesso.

Machine Learning e IA são úteis porque conseguem aprender características faciais relevantes.

Em vez de comparar apenas pixels, o FaceNet transforma cada rosto em um embedding facial: um vetor numérico que representa características importantes da face.

# 3. Objetivo da aplicação

Comparar uma foto de documento com uma foto atual do usuário.

A aplicação deve responder:

- `MATCH`: provavelmente é a mesma pessoa.
- `NO MATCH`: provavelmente não é a mesma pessoa.
- `ERRO`: não foi possível concluir a comparação.

# 4. Bibliotecas usadas

Esta célula instala as dependências principais.

Execute apenas se o ambiente ainda não tiver os pacotes instalados.

In [ ]:
# Biblioteca base para cálculos numéricos e arrays
%pip install numpy

# Biblioteca para gráficos e visualizações
%pip install matplotlib

# Biblioteca para processamento de imagens e vídeo
%pip install opencv-python

# Biblioteca para abrir, salvar e manipular imagens
%pip install pillow

# Detector de rostos baseado em redes neurais
%pip install mtcnn

# Modelo FaceNet pronto para extrair embeddings faciais
%pip install keras-facenet

# Biblioteca de machine learning, usada para classificação e métricas
%pip install scikit-learn

# Framework de deep learning usado pelo FaceNet/Keras
%pip install tensorflow

In [ ]:
from pathlib import Path
import contextlib
import io
import os
import warnings

# Roda em CPU e reduz avisos internos do TensorFlow que parecem erro.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
warnings.filterwarnings("ignore")

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps

MTCNN = None
FaceNet = None


# 5. Configuração inicial

Altere os caminhos abaixo para apontar para as imagens usadas na demonstração.

O limite de distância controla a decisão final:

- distância menor que o limite: `MATCH`;
- distância maior ou igual ao limite: `NO MATCH`.

In [ ]:
caminho_imagem_documento = "Input/documento.jpg"
caminho_imagem_usuario = "Input/usuario5.jpg"

# Para documento x selfie, 1.00 é menos rígido que 0.90 e reduz falso NO MATCH.
limite_distancia_match = 1.00
tamanho_imagem_facenet = (160, 160)
confianca_minima_rosto = 0.90
margem_recorte_rosto = 0.25

# Se quiser capturar a foto atual pela webcam, altere para True.
# Em alguns ambientes Jupyter/Colab, a webcam pode não estar disponível.
usar_webcam_para_usuario = False
indice_camera_webcam = 0


# 6. Carregamento das imagens

A aplicação aceita imagens por caminho de arquivo.

Também existe uma função opcional para capturar imagem pela webcam, se o ambiente permitir.

In [ ]:
def carregar_imagem_de_arquivo(caminho_imagem):
    caminho_imagem = Path(caminho_imagem)

    if not caminho_imagem.exists():
        raise FileNotFoundError(f"ERRO: arquivo não encontrado: {caminho_imagem}")

    try:
        with Image.open(caminho_imagem) as imagem_pil:
            imagem_corrigida = ImageOps.exif_transpose(imagem_pil)
            imagem_rgb = np.array(imagem_corrigida.convert("RGB"))
    except Exception as erro_imagem:
        raise ValueError(
            f"ERRO: imagem inválida ou formato não suportado: {caminho_imagem}"
        ) from erro_imagem

    return imagem_rgb


def capturar_imagem_da_webcam(indice_camera=0):
    captura_webcam = cv2.VideoCapture(indice_camera)

    if not captura_webcam.isOpened():
        raise RuntimeError("ERRO: não foi possível acessar a webcam.")

    captura_realizada, frame_bgr = captura_webcam.read()
    captura_webcam.release()

    if not captura_realizada or frame_bgr is None:
        raise RuntimeError("ERRO: não foi possível capturar imagem pela webcam.")

    imagem_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    return imagem_rgb


def carregar_imagens_para_comparacao(caminho_documento, caminho_usuario, usar_webcam=False):
    imagem_documento = carregar_imagem_de_arquivo(caminho_documento)

    if usar_webcam:
        imagem_usuario = capturar_imagem_da_webcam(indice_camera_webcam)
    else:
        imagem_usuario = carregar_imagem_de_arquivo(caminho_usuario)

    return imagem_documento, imagem_usuario


In [ ]:
try:
    imagem_documento, imagem_usuario = carregar_imagens_para_comparacao(
        caminho_imagem_documento,
        caminho_imagem_usuario,
        usar_webcam=usar_webcam_para_usuario,
    )
    print("Imagens carregadas com sucesso.")
except Exception as erro_carregamento:
    imagem_documento = None
    imagem_usuario = None
    print(erro_carregamento)
    print("Ajuste os caminhos das imagens antes de continuar a execução completa.")

# 7. Exibição das imagens

Aqui as imagens são exibidas para conferência visual.

In [ ]:
def exibir_imagens_lado_a_lado(imagem_documento, imagem_usuario):
    if imagem_documento is None or imagem_usuario is None:
        print("ERRO: carregue as duas imagens antes de exibir.")
        return

    figura, eixos = plt.subplots(1, 2, figsize=(10, 5))

    eixos[0].imshow(imagem_documento)
    eixos[0].set_title("Foto do documento")
    eixos[0].axis("off")

    eixos[1].imshow(imagem_usuario)
    eixos[1].set_title("Foto atual do usuário")
    eixos[1].axis("off")

    plt.tight_layout()
    plt.show()


exibir_imagens_lado_a_lado(imagem_documento, imagem_usuario)

# 8. Detecção de rosto

A função abaixo detecta o rosto na imagem e recorta a região facial.

Se mais de um rosto for encontrado, a função usa o maior rosto detectado e mostra um aviso.

In [ ]:
def carregar_modelos_faciais():
    global MTCNN, FaceNet

    if MTCNN is None or FaceNet is None:
        try:
            # Importa bibliotecas pesadas aqui para reduzir avisos visuais no notebook.
            with contextlib.redirect_stderr(io.StringIO()):
                from mtcnn import MTCNN as ClasseMTCNN
                from keras_facenet import FaceNet as ClasseFaceNet

            MTCNN = ClasseMTCNN
            FaceNet = ClasseFaceNet
        except ImportError as erro_importacao:
            raise RuntimeError(
                "ERRO: modelo FaceNet não carregado. Instale mtcnn, keras-facenet e tensorflow."
            ) from erro_importacao

    with contextlib.redirect_stderr(io.StringIO()):
        detector_rosto = MTCNN()
        modelo_facenet = FaceNet()

    return detector_rosto, modelo_facenet


try:
    detector_rosto, modelo_facenet = carregar_modelos_faciais()
    print("Detector de rosto e modelo FaceNet carregados com sucesso.")
except Exception as erro_modelo:
    detector_rosto = None
    modelo_facenet = None
    print(erro_modelo)


In [ ]:
def detectar_e_recortar_rosto(
    imagem_original,
    detector_rosto,
    confianca_minima=0.90,
    margem_rosto=0.25,
):
    if imagem_original is None:
        raise ValueError("ERRO: imagem não carregada.")

    if detector_rosto is None:
        raise RuntimeError("ERRO: detector de rosto não carregado.")

    deteccoes_rosto = detector_rosto.detect_faces(imagem_original)
    deteccoes_confiaveis = [
        deteccao_rosto
        for deteccao_rosto in deteccoes_rosto
        if deteccao_rosto.get("confidence", 0) >= confianca_minima
    ]

    if len(deteccoes_confiaveis) == 0:
        raise RuntimeError("ERRO: nenhum rosto foi detectado na imagem.")

    if len(deteccoes_confiaveis) > 1:
        print("ATENÇÃO: mais de um rosto detectado. O maior rosto será usado.")

    deteccao_principal = max(
        deteccoes_confiaveis,
        key=lambda deteccao_rosto: deteccao_rosto["box"][2] * deteccao_rosto["box"][3],
    )

    coordenada_esquerda, coordenada_superior, largura_rosto, altura_rosto = deteccao_principal["box"]

    margem_horizontal = int(largura_rosto * margem_rosto)
    margem_vertical = int(altura_rosto * margem_rosto)

    coordenada_direita_original = coordenada_esquerda + largura_rosto
    coordenada_inferior_original = coordenada_superior + altura_rosto

    coordenada_esquerda = max(0, coordenada_esquerda - margem_horizontal)
    coordenada_superior = max(0, coordenada_superior - margem_vertical)
    coordenada_direita = min(imagem_original.shape[1], coordenada_direita_original + margem_horizontal)
    coordenada_inferior = min(imagem_original.shape[0], coordenada_inferior_original + margem_vertical)

    rosto_recortado = imagem_original[
        coordenada_superior:coordenada_inferior,
        coordenada_esquerda:coordenada_direita,
    ]

    if rosto_recortado.size == 0:
        raise RuntimeError("ERRO: rosto detectado, mas o recorte ficou vazio.")

    return rosto_recortado, deteccao_principal


In [ ]:
def exibir_rostos_recortados(rosto_documento, rosto_usuario):
    figura, eixos = plt.subplots(1, 2, figsize=(8, 4))

    eixos[0].imshow(rosto_documento)
    eixos[0].set_title("Rosto do documento")
    eixos[0].axis("off")

    eixos[1].imshow(rosto_usuario)
    eixos[1].set_title("Rosto do usuário")
    eixos[1].axis("off")

    plt.tight_layout()
    plt.show()


try:
    rosto_documento, deteccao_documento = detectar_e_recortar_rosto(
        imagem_documento,
        detector_rosto,
        confianca_minima=confianca_minima_rosto,
        margem_rosto=margem_recorte_rosto,
    )
    rosto_usuario, deteccao_usuario = detectar_e_recortar_rosto(
        imagem_usuario,
        detector_rosto,
        confianca_minima=confianca_minima_rosto,
        margem_rosto=margem_recorte_rosto,
    )
    exibir_rostos_recortados(rosto_documento, rosto_usuario)
except Exception as erro_deteccao:
    rosto_documento = None
    rosto_usuario = None
    print(erro_deteccao)


# 9. Pré-processamento do rosto

Antes de enviar o rosto para o FaceNet, a imagem precisa ser preparada.

A função abaixo:

- redimensiona o rosto para `160x160`;
- converte para `float32`;
- normaliza os valores;
- adiciona a dimensão de lote esperada pelo modelo.

In [ ]:
def preparar_rosto_para_facenet(rosto_recortado, tamanho_imagem=(160, 160)):
    if rosto_recortado is None:
        raise ValueError("ERRO: rosto recortado não disponível.")

    # Mantém a imagem em RGB. A normalização correta será feita pelo keras-facenet.
    rosto_preparado = cv2.resize(rosto_recortado, tamanho_imagem)
    return rosto_preparado


try:
    rosto_documento_pre_processado = preparar_rosto_para_facenet(
        rosto_documento,
        tamanho_imagem=tamanho_imagem_facenet,
    )
    rosto_usuario_pre_processado = preparar_rosto_para_facenet(
        rosto_usuario,
        tamanho_imagem=tamanho_imagem_facenet,
    )
    print("Rostos preparados com sucesso.")
except Exception as erro_pre_processamento:
    rosto_documento_pre_processado = None
    rosto_usuario_pre_processado = None
    print(erro_pre_processamento)


# 10. Extração dos embeddings com FaceNet

Embedding facial é uma representação numérica do rosto.

O FaceNet transforma a imagem do rosto em um vetor de características.

Depois, comparamos os vetores das duas imagens para estimar se são da mesma pessoa.

In [ ]:
def extrair_embedding_facial(rosto_pre_processado, modelo_facenet):
    if rosto_pre_processado is None:
        raise ValueError("ERRO: rosto preparado não disponível.")

    if modelo_facenet is None:
        raise RuntimeError("ERRO: modelo FaceNet não carregado.")

    if not hasattr(modelo_facenet, "model"):
        raise RuntimeError("ERRO: objeto FaceNet inválido.")

    tamanho_modelo = modelo_facenet.metadata.get("image_size", tamanho_imagem_facenet[0])
    rosto_redimensionado = cv2.resize(rosto_pre_processado, (tamanho_modelo, tamanho_modelo))

    if hasattr(modelo_facenet, "_normalize"):
        rosto_normalizado = modelo_facenet._normalize(rosto_redimensionado)
    else:
        rosto_float = rosto_redimensionado.astype("float32")
        media_pixels = rosto_float.mean()
        desvio_pixels = rosto_float.std()
        rosto_normalizado = (rosto_float - media_pixels) / desvio_pixels

    lote_rosto = np.float32([rosto_normalizado])
    embedding_facial = modelo_facenet.model.predict(lote_rosto, verbose=0)[0]
    return embedding_facial


try:
    embedding_documento = extrair_embedding_facial(rosto_documento_pre_processado, modelo_facenet)
    embedding_usuario = extrair_embedding_facial(rosto_usuario_pre_processado, modelo_facenet)
    print("Embeddings extraídos com sucesso.")
    print(f"Tamanho do embedding do documento: {embedding_documento.shape[0]}")
    print(f"Tamanho do embedding do usuário: {embedding_usuario.shape[0]}")
except Exception as erro_embedding:
    embedding_documento = None
    embedding_usuario = None
    print(erro_embedding)


# 11. Comparação dos embeddings

A comparação usa distância euclidiana.

Regra usada:

- distância menor que o limite: `MATCH`;
- distância maior ou igual ao limite: `NO MATCH`.

In [ ]:
def calcular_similaridade_cosseno(embedding_documento, embedding_usuario):
    norma_documento = np.linalg.norm(embedding_documento)
    norma_usuario = np.linalg.norm(embedding_usuario)

    if norma_documento == 0 or norma_usuario == 0:
        return 0.0

    similaridade_cosseno = np.dot(embedding_documento, embedding_usuario) / (
        norma_documento * norma_usuario
    )
    return float(similaridade_cosseno)


def comparar_embeddings_faciais(embedding_documento, embedding_usuario, limite_distancia_match):
    if embedding_documento is None or embedding_usuario is None:
        raise ValueError("ERRO: embeddings faciais não disponíveis.")

    distancia_calculada = float(np.linalg.norm(embedding_documento - embedding_usuario))
    similaridade_cosseno = calcular_similaridade_cosseno(
        embedding_documento,
        embedding_usuario,
    )
    diferenca_para_limite = float(limite_distancia_match - distancia_calculada)
    percentual_do_limite = float((distancia_calculada / limite_distancia_match) * 100)

    if distancia_calculada <= limite_distancia_match:
        resultado_match = "CORRESPONDÊNCIA"
        mensagem_resultado = (
            "A foto do documento e a foto atual provavelmente são da mesma pessoa."
        )
    else:
        resultado_match = "SEM CORRESPONDÊNCIA"
        mensagem_resultado = (
            "A foto do documento e a foto atual provavelmente não são da mesma pessoa."
        )

    atributos_comparacao = {
        "distancia_euclidiana": distancia_calculada,
        "limite_distancia": float(limite_distancia_match),
        "diferenca_para_limite": diferenca_para_limite,
        "percentual_do_limite": percentual_do_limite,
        "similaridade_cosseno": similaridade_cosseno,
    }

    return distancia_calculada, resultado_match, mensagem_resultado, atributos_comparacao


try:
    distancia_calculada, resultado_match, mensagem_resultado, atributos_comparacao = comparar_embeddings_faciais(
        embedding_documento,
        embedding_usuario,
        limite_distancia_match,
    )
    print("Comparação concluída.")
except Exception as erro_comparacao:
    distancia_calculada = None
    resultado_match = "ERRO"
    mensagem_resultado = str(erro_comparacao)
    atributos_comparacao = {}
    print(erro_comparacao)


# 12. Resultado final

Esta etapa mostra a decisão final da aplicação de forma simples.

In [ ]:
def formatar_numero_comparacao(valor_numerico, casas_decimais=4):
    if valor_numerico is None:
        return "não disponível"

    return f"{valor_numerico:.{casas_decimais}f}"


def exibir_resultado_final(resultado_match, distancia_calculada, limite_distancia_match, mensagem_resultado):
    print(f"Resultado: {resultado_match}")

    if distancia_calculada is not None:
        print(f"Distância euclidiana: {distancia_calculada:.4f}")

    print(f"Limite configurado: {limite_distancia_match:.2f}")
    print(f"Conclusão: {mensagem_resultado}")


def criar_figura_relatorio_comparacao(
    imagem_documento,
    imagem_usuario,
    rosto_documento,
    rosto_usuario,
    resultado_match,
    mensagem_resultado,
    atributos_comparacao,
):
    resultado_cor = "#15803d" if resultado_match == "CORRESPONDÊNCIA" else "#b91c1c"

    figura = plt.figure(figsize=(12, 7))
    grade = figura.add_gridspec(
        2,
        3,
        width_ratios=[1, 1, 0.9],
        height_ratios=[1, 1],
        wspace=0.16,
        hspace=0.24,
    )

    eixo_documento = figura.add_subplot(grade[0, 0])
    eixo_usuario = figura.add_subplot(grade[0, 1])
    eixo_rosto_documento = figura.add_subplot(grade[1, 0])
    eixo_rosto_usuario = figura.add_subplot(grade[1, 1])
    eixo_resultado = figura.add_subplot(grade[:, 2])

    eixo_documento.imshow(imagem_documento)
    eixo_documento.set_title("Foto do documento")
    eixo_documento.axis("off")

    eixo_usuario.imshow(imagem_usuario)
    eixo_usuario.set_title("Foto atual")
    eixo_usuario.axis("off")

    eixo_rosto_documento.imshow(rosto_documento)
    eixo_rosto_documento.set_title("Rosto do documento")
    eixo_rosto_documento.axis("off")

    eixo_rosto_usuario.imshow(rosto_usuario)
    eixo_rosto_usuario.set_title("Rosto atual")
    eixo_rosto_usuario.axis("off")

    distancia_euclidiana = atributos_comparacao.get("distancia_euclidiana")
    limite_distancia = atributos_comparacao.get("limite_distancia")
    diferenca_para_limite = atributos_comparacao.get("diferenca_para_limite")
    percentual_do_limite = atributos_comparacao.get("percentual_do_limite")
    similaridade_cosseno = atributos_comparacao.get("similaridade_cosseno")
    confianca_rosto_documento = atributos_comparacao.get("confianca_rosto_documento")
    confianca_rosto_usuario = atributos_comparacao.get("confianca_rosto_usuario")

    linhas_atributos = [
        "Resultado",
        resultado_match,
        "",
        "Distância euclidiana",
        formatar_numero_comparacao(distancia_euclidiana),
        "",
        "Limite de decisão",
        formatar_numero_comparacao(limite_distancia, 2),
        "",
        "Diferença para o limite",
        formatar_numero_comparacao(diferenca_para_limite),
        "",
        "Uso do limite",
        f"{formatar_numero_comparacao(percentual_do_limite, 2)}%",
        "",
        "Similaridade cosseno",
        formatar_numero_comparacao(similaridade_cosseno),
    ]

    if confianca_rosto_documento is not None and confianca_rosto_usuario is not None:
        linhas_atributos.extend([
            "",
            "Confiança do detector",
            f"Documento: {formatar_numero_comparacao(confianca_rosto_documento)}",
            f"Foto atual: {formatar_numero_comparacao(confianca_rosto_usuario)}",
        ])

    linhas_atributos.extend([
        "",
        "Conclusão",
        mensagem_resultado,
    ])

    texto_atributos = "\n".join(linhas_atributos)

    eixo_resultado.axis("off")
    eixo_resultado.text(
        0.02,
        0.98,
        texto_atributos,
        transform=eixo_resultado.transAxes,
        va="top",
        ha="left",
        fontsize=11,
        linespacing=1.25,
        color="#111827",
        wrap=True,
    )
    eixo_resultado.text(
        0.02,
        1.05,
        resultado_match,
        transform=eixo_resultado.transAxes,
        va="bottom",
        ha="left",
        fontsize=16,
        fontweight="bold",
        color=resultado_cor,
    )

    figura.suptitle("Relatório de comparação facial", fontsize=16, fontweight="bold")
    return figura


def exibir_relatorio_comparacao(
    imagem_documento,
    imagem_usuario,
    rosto_documento,
    rosto_usuario,
    resultado_match,
    mensagem_resultado,
    atributos_comparacao,
):
    figura = criar_figura_relatorio_comparacao(
        imagem_documento,
        imagem_usuario,
        rosto_documento,
        rosto_usuario,
        resultado_match,
        mensagem_resultado,
        atributos_comparacao,
    )
    plt.show()

    exibir_resultado_final(
        resultado_match,
        atributos_comparacao.get("distancia_euclidiana"),
        atributos_comparacao.get("limite_distancia"),
        mensagem_resultado,
    )


exibir_relatorio_comparacao(
    imagem_documento,
    imagem_usuario,
    rosto_documento,
    rosto_usuario,
    resultado_match,
    mensagem_resultado,
    atributos_comparacao,
)


# 13. Demonstração completa

A função abaixo executa o fluxo completo em uma única chamada.

Etapas executadas:

1. carrega imagem do documento;
2. carrega imagem do usuário;
3. detecta os rostos;
4. pré-processa os rostos;
5. extrai embeddings;
6. compara embeddings;
7. exibe resultado.

In [ ]:
def executar_photo_match(
    caminho_imagem_documento,
    caminho_imagem_usuario,
    limite_distancia_match=1.00,
    usar_webcam=False,
):
    try:
        detector_rosto_local, modelo_facenet_local = carregar_modelos_faciais()

        imagem_documento_local, imagem_usuario_local = carregar_imagens_para_comparacao(
            caminho_imagem_documento,
            caminho_imagem_usuario,
            usar_webcam=usar_webcam,
        )

        rosto_documento_local, deteccao_documento_local = detectar_e_recortar_rosto(
            imagem_documento_local,
            detector_rosto_local,
            confianca_minima=confianca_minima_rosto,
            margem_rosto=margem_recorte_rosto,
        )
        rosto_usuario_local, deteccao_usuario_local = detectar_e_recortar_rosto(
            imagem_usuario_local,
            detector_rosto_local,
            confianca_minima=confianca_minima_rosto,
            margem_rosto=margem_recorte_rosto,
        )

        rosto_documento_pre_processado_local = preparar_rosto_para_facenet(
            rosto_documento_local,
            tamanho_imagem=tamanho_imagem_facenet,
        )
        rosto_usuario_pre_processado_local = preparar_rosto_para_facenet(
            rosto_usuario_local,
            tamanho_imagem=tamanho_imagem_facenet,
        )

        embedding_documento_local = extrair_embedding_facial(
            rosto_documento_pre_processado_local,
            modelo_facenet_local,
        )
        embedding_usuario_local = extrair_embedding_facial(
            rosto_usuario_pre_processado_local,
            modelo_facenet_local,
        )

        distancia_calculada_local, resultado_match_local, mensagem_resultado_local, atributos_comparacao_local = comparar_embeddings_faciais(
            embedding_documento_local,
            embedding_usuario_local,
            limite_distancia_match,
        )

        atributos_comparacao_local["confianca_rosto_documento"] = float(
            deteccao_documento_local.get("confidence", 0)
        )
        atributos_comparacao_local["confianca_rosto_usuario"] = float(
            deteccao_usuario_local.get("confidence", 0)
        )

        exibir_relatorio_comparacao(
            imagem_documento_local,
            imagem_usuario_local,
            rosto_documento_local,
            rosto_usuario_local,
            resultado_match_local,
            mensagem_resultado_local,
            atributos_comparacao_local,
        )

        return {
            "resultado": resultado_match_local,
            "mensagem": mensagem_resultado_local,
            "atributos_comparacao": atributos_comparacao_local,
        }

    except FileNotFoundError as erro_arquivo:
        print(erro_arquivo)
        return {"resultado": "ERRO", "mensagem": str(erro_arquivo)}
    except ValueError as erro_valor:
        print(erro_valor)
        return {"resultado": "ERRO", "mensagem": str(erro_valor)}
    except RuntimeError as erro_execucao:
        print(erro_execucao)
        return {"resultado": "ERRO", "mensagem": str(erro_execucao)}
    except Exception as erro_inesperado:
        print(f"ERRO: falha inesperada: {erro_inesperado}")
        return {"resultado": "ERRO", "mensagem": str(erro_inesperado)}


In [ ]:
# Execute depois de ajustar os caminhos das imagens.

resultado_photo_match = executar_photo_match(
    caminho_imagem_documento,
    caminho_imagem_usuario,
    limite_distancia_match=limite_distancia_match,
    usar_webcam=usar_webcam_para_usuario,
)

resultado_photo_match

# 14. Comparação em lote e salvamento dos relatórios

Esta etapa compara `Input/documento.jpg` com todas as imagens `Input/usuario*.jpg` e salva um relatório visual em `Output/`.


In [ ]:
def ordenar_arquivos_usuario(caminho_usuario):
    nome_arquivo = caminho_usuario.stem
    numero_texto = "".join(caractere for caractere in nome_arquivo if caractere.isdigit())

    if numero_texto:
        return int(numero_texto)

    return 0


def comparar_documento_com_usuario(
    imagem_documento,
    caminho_imagem_usuario,
    detector_rosto,
    modelo_facenet,
    limite_distancia_match=1.00,
):
    imagem_usuario = carregar_imagem_de_arquivo(caminho_imagem_usuario)

    rosto_documento, deteccao_documento = detectar_e_recortar_rosto(
        imagem_documento,
        detector_rosto,
        confianca_minima=confianca_minima_rosto,
        margem_rosto=margem_recorte_rosto,
    )
    rosto_usuario, deteccao_usuario = detectar_e_recortar_rosto(
        imagem_usuario,
        detector_rosto,
        confianca_minima=confianca_minima_rosto,
        margem_rosto=margem_recorte_rosto,
    )

    rosto_documento_pre_processado = preparar_rosto_para_facenet(
        rosto_documento,
        tamanho_imagem=tamanho_imagem_facenet,
    )
    rosto_usuario_pre_processado = preparar_rosto_para_facenet(
        rosto_usuario,
        tamanho_imagem=tamanho_imagem_facenet,
    )

    embedding_documento = extrair_embedding_facial(
        rosto_documento_pre_processado,
        modelo_facenet,
    )
    embedding_usuario = extrair_embedding_facial(
        rosto_usuario_pre_processado,
        modelo_facenet,
    )

    distancia_calculada, resultado_match, mensagem_resultado, atributos_comparacao = comparar_embeddings_faciais(
        embedding_documento,
        embedding_usuario,
        limite_distancia_match,
    )

    atributos_comparacao["confianca_rosto_documento"] = float(
        deteccao_documento.get("confidence", 0)
    )
    atributos_comparacao["confianca_rosto_usuario"] = float(
        deteccao_usuario.get("confidence", 0)
    )

    return {
        "imagem_usuario": imagem_usuario,
        "rosto_documento": rosto_documento,
        "rosto_usuario": rosto_usuario,
        "resultado": resultado_match,
        "mensagem": mensagem_resultado,
        "atributos_comparacao": atributos_comparacao,
    }


def executar_photo_match_em_lote(
    caminho_documento="Input/documento.jpg",
    padrao_usuarios="Input/usuario*.jpg",
    pasta_saida="Output",
    limite_distancia_match=1.00,
):
    caminho_documento = Path(caminho_documento)
    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    caminhos_usuarios = sorted(
        Path().glob(padrao_usuarios),
        key=ordenar_arquivos_usuario,
    )

    if not caminhos_usuarios:
        raise FileNotFoundError(f"ERRO: nenhuma imagem encontrada no padrão: {padrao_usuarios}")

    detector_rosto, modelo_facenet = carregar_modelos_faciais()
    imagem_documento = carregar_imagem_de_arquivo(caminho_documento)
    resultados_lote = []

    for caminho_usuario_atual in caminhos_usuarios:
        print(f"Comparando {caminho_documento.name} com {caminho_usuario_atual.name}...")

        try:
            resultado_comparacao = comparar_documento_com_usuario(
                imagem_documento,
                caminho_usuario_atual,
                detector_rosto,
                modelo_facenet,
                limite_distancia_match=limite_distancia_match,
            )

            nome_saida = f"resultado_{caminho_usuario_atual.stem}.png"
            caminho_saida = pasta_saida / nome_saida

            figura = criar_figura_relatorio_comparacao(
                imagem_documento,
                resultado_comparacao["imagem_usuario"],
                resultado_comparacao["rosto_documento"],
                resultado_comparacao["rosto_usuario"],
                resultado_comparacao["resultado"],
                resultado_comparacao["mensagem"],
                resultado_comparacao["atributos_comparacao"],
            )
            figura.savefig(caminho_saida, dpi=160, bbox_inches="tight")
            plt.close(figura)

            resultados_lote.append({
                "arquivo_usuario": str(caminho_usuario_atual),
                "arquivo_saida": str(caminho_saida),
                "resultado": resultado_comparacao["resultado"],
                "mensagem": resultado_comparacao["mensagem"],
                "atributos_comparacao": resultado_comparacao["atributos_comparacao"],
            })

            distancia = resultado_comparacao["atributos_comparacao"]["distancia_euclidiana"]
            print(f"Resultado: {resultado_comparacao['resultado']} | Distância: {distancia:.4f}")
            print(f"Relatório salvo em: {caminho_saida}")

        except Exception as erro_comparacao:
            resultados_lote.append({
                "arquivo_usuario": str(caminho_usuario_atual),
                "resultado": "ERRO",
                "mensagem": str(erro_comparacao),
            })
            print(f"ERRO ao comparar {caminho_usuario_atual.name}: {erro_comparacao}")

    return resultados_lote


resultados_lote = executar_photo_match_em_lote(
    caminho_documento="Input/documento.jpg",
    padrao_usuarios="Input/usuario*.jpg",
    pasta_saida="Output",
    limite_distancia_match=limite_distancia_match,
)

resultados_lote


# 14. Limpeza antes de subir para o GitHub

- Execute esta célula como última etapa antes de salvar/commitar o notebook.
- Ela remove resultados salvos no notebook e arquivos sensíveis locais.


In [34]:
# 17. Limpeza antes de subir para o GitHub
# Execute esta célula como última etapa antes de salvar/commitar o notebook.

from pathlib import Path
import gc
import json
import shutil
from IPython.display import clear_output

# Arquivo do próprio notebook que será limpo.
caminho_notebook = Path("Univali_M3_FaceNet.ipynb")

# Pastas usadas para documento, selfie, imagens temporárias e resultados.
pastas_com_arquivos_sensiveis = [Path("Input"), Path("Output")]

# Palavras usadas para localizar variáveis com imagem, rosto, documento ou embedding em memória.
palavras_sensiveis_em_memoria = (
    "imagem",
    "rosto",
    "embedding",
    "deteccao",
    "documento",
    "usuario",
    "photo_match",
    "distancia",
)

# Remove da memória variáveis que podem conter identidade, documento, rosto ou resultado.
nomes_variaveis_sensiveis = [
    nome_variavel
    for nome_variavel in list(globals())
    if any(
        palavra_sensivel in nome_variavel.lower()
        for palavra_sensivel in palavras_sensiveis_em_memoria
    )
]

for nome_variavel in nomes_variaveis_sensiveis:
    globals().pop(nome_variavel, None)

# Remove arquivos das pastas sensíveis, mantendo apenas o .keep para preservar a pasta no Git.
for pasta_sensivel in pastas_com_arquivos_sensiveis:
    pasta_sensivel.mkdir(exist_ok=True)

    for caminho_sensivel in pasta_sensivel.iterdir():
        if caminho_sensivel.name == ".keep":
            continue

        if caminho_sensivel.is_dir():
            shutil.rmtree(caminho_sensivel)
        else:
            caminho_sensivel.unlink(missing_ok=True)

# Remove outputs, imagens renderizadas, prints e contadores de execução salvos no .ipynb.
if caminho_notebook.exists():
    conteudo_notebook = json.loads(caminho_notebook.read_text(encoding="utf-8"))

    for celula_notebook in conteudo_notebook.get("cells", []):
        if celula_notebook.get("cell_type") == "code":
            celula_notebook["outputs"] = []
            celula_notebook["execution_count"] = None

    caminho_notebook.write_text(
        json.dumps(conteudo_notebook, ensure_ascii=False, indent=1) + "\n",
        encoding="utf-8",
    )

# Força a coleta de lixo para liberar objetos grandes da memória.
gc.collect()

# Limpa também a saída visual desta própria célula.
clear_output(wait=True)
